# 22  — User Profile Construction

This notebook defines and validates the **UserProfile entrypoint** for Chapter 3 of the Job Intelligence Engine. Its purpose is to formalise how messy, free-form user inputs (skills text, location, and optional preferences) are transformed into a deterministic, model-ready representation that can be safely consumed by all downstream components.

The notebook composes validated artefacts from earlier chapters: Chapter 0 skill extraction and explanation logic, and the Chapter 1 skills PCA transformer. These components are reused without modification to ensure full alignment between user-side and job-side representations in the shared skill space.

The resulting UserProfile schema acts as the single source of truth for individual positioning in the system. All subsequent Chapter 3 analyses—suitability scoring, skill gap analysis, competitiveness estimation, sensitivity analysis, and future public APIs—must consume this output and must not re-implement user parsing, validation, or preprocessing logic elsewhere.


## Set Up

### Libraries

In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
#===
import sys
from pathlib import Path

### Path

In [4]:
project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))
project_root

PosixPath('/Users/alejandrofp/Desktop/Projects/03_Flagship_Portfolio/job-intelligence-engine')

In [5]:
from src.job_intel.features.userprofile_skill_processing import user_skill_processor
from src.job_intel.config import CH2_PROCESSED_DF

### Data

In [6]:

df = pd.read_csv(CH2_PROCESSED_DF)

for col in df.columns:
    print(col)

job_id
Job Description
Rating
Size
Founded
Industry
Sector
role_source
state
ownership_clean
job_description_clean
job_title_base
seniority_combined
job_title_norm
job_title_family
domain
sal_is_hourly
sal_min
sal_max
sal_mean
title_plus_description
core_programming__basic
core_programming__intermediate
core_programming__advanced
data_engineering_pipelines__basic
data_engineering_pipelines__intermediate
data_engineering_pipelines__advanced
ml_ai__basic
ml_ai__intermediate
ml_ai__advanced
analytics_stats__basic
analytics_stats__intermediate
analytics_stats__advanced
bi_viz__basic
bi_viz__intermediate
bi_viz__advanced
cloud__basic
cloud__intermediate
cloud__advanced
db_storage__basic
db_storage__intermediate
db_storage__advanced
productivity_workflow__basic
productivity_workflow__intermediate
productivity_workflow__advanced
soft_skills__core
soft_skills__leadership
domain_specific__none
Size_1 to 50 employees
Size_10000+ employees
Size_1001 to 5000 employees
Size_201 to 500 employees
Siz

## Features lookup

In [13]:
size_lookup = df[['Size', 'size_code']].drop_duplicates()
sector_lookup = df[['Sector', 'sector_code']].drop_duplicates()
state_lookup = df[['state', 'state_code']].drop_duplicates()
ownership_lookup = df[['ownership_clean', 'ownership_code']].drop_duplicates()
seniority_lookup = df[['seniority_combined', 'seniority_code']].drop_duplicates()
title_lookup = df[['title_rich', 'title_rich_code']].drop_duplicates()
title_family_lookup = df[['job_title_family', 'title_code']].drop_duplicates()

## User profile builder

In [18]:
from datetime import datetime, timezone
from typing import Optional, Union, List, Dict, Any

from src.job_intel.features.userprofile_skill_processing import user_skill_processor


# ------------------------------------------------------------------
# Frozen allow-lists (sourced from Chapter 1 training data)
# ------------------------------------------------------------------

ALLOWED_SECTORS = [
    "Travel & Tourism",
    "Consumer Services",
    "Unknown",
    "Information Technology",
    "Business Services",
    "Insurance",
    "Finance",
    "Retail",
    "Media",
    "Restaurants, Bars & Food Services",
    "Agriculture & Forestry",
    "Non-Profit",
    "Education",
    "Government",
    "Health Care",
    "Oil, Gas, Energy & Utilities",
    "Accounting & Legal",
    "Manufacturing",
    "Real Estate",
    "Biotech & Pharmaceuticals",
    "Arts, Entertainment & Recreation",
    "Aerospace & Defense",
    "Construction, Repair & Maintenance",
    "Transportation & Logistics",
    "Telecommunications",
    "Mining & Metals",
]

ALLOWED_STATES = {
    "NY",
    "NJ",
    "CA",
    "IL",
    "TX",
    "AZ",
    "DE",
    "PA",
    "FL",
    "OH",
    "UT",
    "VA",
    "NC",
    "SC",
    "IN",
    "WA",
    "GA",
    "KS",
    "CO",
    "international",
}

ALLOWED_TITLE_RICH = [
    "general_data_data_scientist",
    "business_data_scientist",
    "general_data_data_analyst",
    "research_scientist",
    "ML_AI_scientist",
    "ML_AI_data_scientist",
    "health_data_engineer",
    "business_data_analyst",
    "general_data_data_engineer",
    "health_data_scientist",
    "ML_AI_data_engineer",
    "business_data_engineer",
    "sport_data_scientist",
    "research_data_analyst",
    "sport_data_analyst",
    "health_scientist",
    "health_data_analyst",
    "research_data_scientist",
    "security_data_scientist",
    "ML_AI_data_analyst",
    "business_scientist",
    "security_data_analyst",
    "security_scientist",
    "general_data_scientist",
    "security_data_engineer",
]

ALLOWED_TITLE_FAMILY = [
    "data_scientist",
    "data_analyst",
    "scientist",
    "data_engineer",
]


# ------------------------------------------------------------------
# UserProfile constructor
# ------------------------------------------------------------------


def build_user_profile(
    skill_text: str = "",
    current_state: Optional[str] = "ALL",
    job_title_family: Optional[str] = None,
    job_title_rich: Optional[str] = None,
    target_sectors: Optional[Union[str, List[str]]] = None,
    salary_target: Optional[Union[int, float, str]] = None,
    explain_skills: bool = False,
    schema_version: str = "v01",
) -> Dict[str, Any]:

    # --------------------------------------------------------------
    # Normalize state
    # --------------------------------------------------------------
    if (
        current_state is None
        or str(current_state).strip() == ""
        or str(current_state).upper() == "ALL"
    ):
        current_state = None
    else:
        state_norm = current_state.strip()
        if state_norm.lower() == "international":
            current_state = "international"
        else:
            current_state = state_norm.upper()

        if current_state not in ALLOWED_STATES:
            raise ValueError(
                f"Unknown state '{current_state}'. Must be one of: {sorted(ALLOWED_STATES)}"
            )

    # --------------------------------------------------------------
    # Normalize and validate title filters
    # --------------------------------------------------------------
    if job_title_family is not None:
        if job_title_family not in ALLOWED_TITLE_FAMILY:
            raise ValueError(
                f"Unknown job_title_family '{job_title_family}'. "
                f"Must be one of: {ALLOWED_TITLE_FAMILY}"
            )

    if job_title_rich is not None:
        if job_title_rich not in ALLOWED_TITLE_RICH:
            raise ValueError(
                f"Unknown job_title_rich '{job_title_rich}'. "
                f"Must be one of: {ALLOWED_TITLE_RICH}"
            )

    # --------------------------------------------------------------
    # Normalize and validate sectors
    # --------------------------------------------------------------
    if isinstance(target_sectors, str):
        target_sectors = [target_sectors]

    if target_sectors is not None:
        if not isinstance(target_sectors, list):
            raise TypeError(
                "target_sectors must be None, a string, or a list of strings"
            )

        target_sectors = [s.strip() for s in target_sectors]

        invalid = [s for s in target_sectors if s not in ALLOWED_SECTORS]
        if invalid:
            raise ValueError(
                f"Unknown sector(s): {invalid}. "
                f"Must be chosen from: {ALLOWED_SECTORS}"
            )

    # --------------------------------------------------------------
    # Salary target validation (stored raw, not used yet)
    # --------------------------------------------------------------
    if salary_target is not None:
        try:
            salary_target = float(salary_target)
        except (TypeError, ValueError) as e:
            raise TypeError("salary_target must be numeric or None") from e

        if salary_target <= 0:
            raise ValueError("salary_target must be > 0")

    # --------------------------------------------------------------
    # Skill processing (validated primitive)
    # --------------------------------------------------------------
    skills = user_skill_processor(skill_text, explain_skills=explain_skills)

    # --------------------------------------------------------------
    # Assemble UserProfile
    # --------------------------------------------------------------
    profile = {
        "raw_inputs": {
            "skill_text": skill_text,
            "current_state": current_state,
            "job_title_family": job_title_family,
            "job_title_rich": job_title_rich,
            "target_sectors": target_sectors,
            "salary_target": salary_target,
        },
        "derived": {
            "skills_by_group": skills["skill_flags_long"],
            "skill_vector": skills["skill_vector"],
            "skill_pcs": skills["skill_pcs"],
            "skill_explanations": skills.get("skill_explanations"),
        },
        "meta": {
            "schema_version": schema_version,
            "created_utc": datetime.now(timezone.utc).isoformat(),
        },
    }

    return profile


### Func tests

#### Test 1 — Normal case

In [19]:
profile = build_user_profile(
    skill_text="Python, pandas, numpy, SQL, scikit-learn, statistics, git",
    current_state="CA",
    explain_skills=True
)




In [20]:

profile["raw_inputs"]

{'skill_text': 'Python, pandas, numpy, SQL, scikit-learn, statistics, git',
 'current_state': 'CA',
 'job_title_family': None,
 'job_title_rich': None,
 'target_sectors': None,
 'salary_target': None}

In [21]:
profile["derived"]["skill_vector"].shape


(1, 27)

In [22]:
profile["derived"]["skill_pcs"].shape

(1, 10)

In [23]:
profile["derived"]["skill_explanations"]

{'core_programming__basic': ['python'],
 'core_programming__intermediate': [],
 'core_programming__advanced': [],
 'data_engineering_pipelines__basic': [],
 'data_engineering_pipelines__intermediate': [],
 'data_engineering_pipelines__advanced': [],
 'ml_ai__basic': ['scikit', 'scikit-learn'],
 'ml_ai__intermediate': [],
 'ml_ai__advanced': [],
 'analytics_stats__basic': ['pandas', 'numpy'],
 'analytics_stats__intermediate': [],
 'analytics_stats__advanced': [],
 'bi_viz__basic': [],
 'bi_viz__intermediate': [],
 'bi_viz__advanced': [],
 'cloud__basic': [],
 'cloud__intermediate': [],
 'cloud__advanced': [],
 'db_storage__basic': ['sql'],
 'db_storage__intermediate': [],
 'db_storage__advanced': [],
 'productivity_workflow__basic': [],
 'productivity_workflow__intermediate': ['git'],
 'productivity_workflow__advanced': [],
 'soft_skills__core': [],
 'soft_skills__leadership': [],
 'domain_specific__none': []}

#### Test 2 — Empty skill text

In [24]:
profile = build_user_profile(skill_text="")

profile["derived"]["skill_pcs"].shape


(1, 10)

In [25]:
profile["derived"]["skill_vector"].sum(axis=1)

0    0
dtype: int64

#### Test 3 — Sparse input

In [26]:
profile = build_user_profile(skill_text="sql")

profile["derived"]["skill_vector"]


,core_programming__basic,core_programming__intermediate,core_programming__advanced,data_engineering_pipelines__basic,data_engineering_pipelines__intermediate,data_engineering_pipelines__advanced,ml_ai__basic,ml_ai__intermediate,ml_ai__advanced,analytics_stats__basic,...,cloud__advanced,db_storage__basic,db_storage__intermediate,db_storage__advanced,productivity_workflow__basic,productivity_workflow__intermediate,productivity_workflow__advanced,soft_skills__core,soft_skills__leadership,domain_specific__none
0,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0


#### Test 4 — Noisy / irrelevant text

In [27]:
profile = build_user_profile(
    skill_text="""
    I enjoy hiking, coffee, long walks on the beach,
    but I also use python for data analysis sometimes.
    """
)

profile["derived"]["skill_pcs"].shape


(1, 10)

In [28]:
profile["derived"]["skill_vector"]

,core_programming__basic,core_programming__intermediate,core_programming__advanced,data_engineering_pipelines__basic,data_engineering_pipelines__intermediate,data_engineering_pipelines__advanced,ml_ai__basic,ml_ai__intermediate,ml_ai__advanced,analytics_stats__basic,...,cloud__advanced,db_storage__basic,db_storage__intermediate,db_storage__advanced,productivity_workflow__basic,productivity_workflow__intermediate,productivity_workflow__advanced,soft_skills__core,soft_skills__leadership,domain_specific__none
0,1,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


#### Test 5 — Filter normalization

In [29]:
profile = build_user_profile(
    skill_text="python",
    current_state="ALL",
    target_sectors="Technology",
    salary_target="120000",
)

profile["raw_inputs"]


ValueError: Unknown sector(s): ['Technology']. Must be chosen from: ['Travel & Tourism', 'Consumer Services', 'Unknown', 'Information Technology', 'Business Services', 'Insurance', 'Finance', 'Retail', 'Media', 'Restaurants, Bars & Food Services', 'Agriculture & Forestry', 'Non-Profit', 'Education', 'Government', 'Health Care', 'Oil, Gas, Energy & Utilities', 'Accounting & Legal', 'Manufacturing', 'Real Estate', 'Biotech & Pharmaceuticals', 'Arts, Entertainment & Recreation', 'Aerospace & Defense', 'Construction, Repair & Maintenance', 'Transportation & Logistics', 'Telecommunications', 'Mining & Metals']

In [30]:
profile = build_user_profile(
    skill_text="python",
    current_state="ALL",
    target_sectors="Information Technology",
    salary_target="120000",
)

profile["raw_inputs"]


{'skill_text': 'python',
 'current_state': None,
 'job_title_family': None,
 'job_title_rich': None,
 'target_sectors': ['Information Technology'],
 'salary_target': 120000.0}

# == End of Notebook ==